# EDA on gold - validate + explore the star schema

## Row counts per gold table

In [0]:
%sql
SELECT 'dim_country' AS tbl, COUNT(*) AS n FROM marathos.gold.dim_country UNION ALL
SELECT 'dim_date',           COUNT(*) FROM marathos.gold.dim_date    UNION ALL
SELECT 'dim_event',          COUNT(*) FROM marathos.gold.dim_event   UNION ALL
SELECT 'dim_athlete',        COUNT(*) FROM marathos.gold.dim_athlete UNION ALL
SELECT 'fct_results',        COUNT(*) FROM marathos.gold.fct_results;

## Grain + referential integrity check

Each dim should have one row per key, and every fct FK should resolve to a dim row.

In [0]:
%sql
WITH grain AS (
  SELECT 'dim_event'   AS tbl, COUNT(*) AS rows, COUNT(DISTINCT event_id)    AS unique_keys FROM marathos.gold.dim_event   UNION ALL
  SELECT 'dim_athlete',        COUNT(*),         COUNT(DISTINCT athlete_id)               FROM marathos.gold.dim_athlete UNION ALL
  SELECT 'dim_country',        COUNT(*),         COUNT(DISTINCT country_id)               FROM marathos.gold.dim_country
),
orphans AS (
  SELECT 'orphan event'   AS tbl, NULL AS rows, COUNT(*) AS unique_keys
    FROM marathos.gold.fct_results f LEFT JOIN marathos.gold.dim_event   e USING (event_id)   WHERE e.event_id IS NULL UNION ALL
  SELECT 'orphan athlete',      NULL,           COUNT(*)
    FROM marathos.gold.fct_results f LEFT JOIN marathos.gold.dim_athlete a USING (athlete_id) WHERE a.athlete_id IS NULL UNION ALL
  SELECT 'orphan country',      NULL,           COUNT(*)
    FROM marathos.gold.fct_results f LEFT JOIN marathos.gold.dim_country c USING (country_id) WHERE c.country_id IS NULL
)
SELECT * FROM grain UNION ALL SELECT * FROM orphans;
-- Grain: rows == unique_keys means 1 row per key (no fan-out).
-- Orphans: unique_keys should be 0 (every fct row finds its dim).

## KPIs

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM marathos.gold.fct_results) AS total_results,
  (SELECT COUNT(*) FROM marathos.gold.dim_athlete) AS unique_athletes,
  (SELECT COUNT(*) FROM marathos.gold.dim_event)   AS unique_events,
  (SELECT COUNT(DISTINCT country_id) FROM marathos.gold.fct_results) AS countries_represented;

## Finishes per year (trend)

Has running grown over the last 20 years? Becomes a dashboard widget.

In [0]:
import plotly.express as px

yearly = spark.sql("""
    SELECT d.year, COUNT(*) AS finishes
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_date d USING (date_id)
    GROUP BY d.year ORDER BY d.year
""").toPandas()

px.line(yearly, x="year", y="finishes",
        title="Ultra-marathon finishes per year", markers=True).show()

In [0]:
%sql
SELECT COUNT(*) AS orphans
   FROM marathos.gold.fct_results f
   LEFT JOIN marathos.gold.dim_country c USING (country_id)
   WHERE c.country_id IS NULL;

## Top 10 countries by finishes

In [0]:
top_countries = spark.sql("""
    SELECT c.country_name, c.continent, COUNT(*) AS finishes
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_country c USING (country_id)
    GROUP BY c.country_name, c.continent
    ORDER BY finishes DESC LIMIT 10
""").toPandas()

fig = px.bar(top_countries, x="finishes", y="country_name",
             color="continent", orientation="h",
             title="Top 10 countries by finishes")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## Distance events - avg finish time per distance

In [0]:
%sql
SELECT
  e.event_value AS distance, e.event_unit,
  COUNT(*) AS finishers,
  ROUND(AVG(f.performance_seconds) / 3600.0, 2) AS avg_hours,
  ROUND(MIN(f.performance_seconds) / 3600.0, 2) AS fastest_hours
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e USING (event_id)
WHERE e.event_type = 'distance' AND e.event_value IN (50, 100)
GROUP BY e.event_value, e.event_unit
ORDER BY e.event_value, e.event_unit;

## Conclusion

Gold validation:
- Grain correct: every dim has exactly one row per key (no fan-out).
- Referential integrity: zero orphan FKs in fct_results across all dims.
- 4-way joins work end-to-end.

Sanity numbers:
- 50km avg 6.8h, 100km avg 13.7h — matches real-world ultra-running pace.
- Top duration distances near the 24h world record (~319 km).
- Yearly trend shows the sport's clear growth post-2010.

Ready for the dashboard and Genie.

## Verify the new event appears

In [0]:
%sql
-- New event should appear in dim_event
SELECT * FROM marathos.gold.dim_event
WHERE event_name = 'Stockholm Ultra Marathon (SWE)';

In [0]:
%sql
-- Should see 250 results in fct_results for the new event
SELECT COUNT(*) FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_event e USING (event_id)
WHERE e.event_name = 'Stockholm Ultra Marathon (SWE)';
